# Initial Data Exploration

Using the Fish Assemblage data 1983 - 2025 what patterns are shown?

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

# Fish Assemblage data
fish_ass = pd.read_excel('./Fish Data.xlsx', na_filter=False)
act_data = pd.read_excel('./Activity Data.xlsx', na_filter=False)

# Filter out records with year = 0 (What is this?)
fish_ass = fish_ass[fish_ass['Year'] != 0]
act_data = act_data[act_data['Year'] != 0]

fish_ass_cols = fish_ass.columns.tolist()
act_cols = act_data.columns.tolist()

print(fish_ass_cols)

print(act_cols)

['ACT_ID', 'Year', 'Date', 'Time', 'Project', 'Waterbody', 'Site_Name', 'Town', 'County', 'Gear', 'Species', 'Length mm', 'Weight g', 'Total_Num', 'Run_Num', 'T_Num_Runs', 'EBT_age_class', 'Clip_obs', 'Clip_admin', 'Scales', 'Age', 'Genetic sample', 'GlobalID', 'Data_Comments', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26']
['ACT_ID', 'Year', 'Date', 'Time', 'Project', 'Water_Body', 'Site_Name', 'Town', 'County', 'Gear', 'Lat_Start', 'Long_Start', 'Lat_End', 'Long_End', 'Site_description', 'HUC_12_NAME', 'HUC_10_NAME', 'HUC_8_NAME', 'HUC_6_NAME', 'SPARROW_ID', 'Gear_size', 'Set_date', 'Set_time', 'Haul_date', 'Haul_time', 'Thermologger ', 'Temp_Act_ID', 'TEMP_LAT', 'TEMP_LONG', 'MJWT', 'MAWT', 'MJAWT', 'Num_Units', 'N_Runs', 'EFISH_time_run1', 'EFISH_time_run2', 'EFISH_time_run3', 'EFISH_time_run4', 'EFISH_time_run5', 'EFISH_time_total', 'EFISH_volts', 'EFISH_freq', 'EFISH_duty', 'EFISH_length', 'EFISH_width1', 'EFISH_width2', 'EFISH_width3', 'EFISH_width4', 'EFISH_width5', 'EFISH_Avg_Wi

In [31]:
combined = act_data[["ACT_ID", "Year", "Lat_Start", "Long_Start", "Water_Body"]].merge(
    fish_ass[["ACT_ID", "Species", "Total_Num"]],
    on="ACT_ID",
    how="left"
)

# Create Site_Num for unique Lat_Start, Long_Start combinations
site_mapping = combined[["Lat_Start", "Long_Start"]].drop_duplicates().reset_index(drop=True)
site_mapping['Site_Num'] = range(1, len(site_mapping) + 1)

combined = combined.merge(site_mapping, on=["Lat_Start", "Long_Start"], how="left")



print(combined.head())

                                          ACT_ID  Year  Lat_Start  Long_Start  \
0  19830721-0-FFF-Piscassic River-Newfields-GILL  1983    43.0343    -70.9681   
1  19830722-0-FFF-Piscassic River-Newfields-GILL  1983    43.0343    -70.9681   
2  19830722-0-FFF-Piscassic River-Newfields-GILL  1983    43.0343    -70.9681   
3  19830723-0-FFF-Piscassic River-Newfields-GILL  1983    43.0343    -70.9681   
4  19830725-0-FFF-Piscassic River-Newfields-GILL  1983    43.0343    -70.9681   

        Water_Body Species Total_Num  Site_Num  
0  Piscassic River     BBH         1         1  
1  Piscassic River     BBH         1         1  
2  Piscassic River     BBH         1         1  
3  Piscassic River     BBH         1         1  
4  Piscassic River     ECP         1         1  


In [ ]:
import requests
import time

# Get elevation for each unique site using OpenTopoData SRTM90m API
unique_sites = combined[["Site_Num", "Lat_Start", "Long_Start"]].drop_duplicates(subset="Site_Num")

def get_elevations(sites_df, batch_size=100):
    elevation_map = {}
    rows = sites_df.to_dict("records")
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        locations = "|".join(f"{r['Lat_Start']},{r['Long_Start']}" for r in batch)
        response = requests.get(f"https://api.opentopodata.org/v1/ned10m?locations={locations}")
        response.raise_for_status()
        results = response.json().get("results", [])
        for r, result in zip(batch, results):
            elevation_map[r["Site_Num"]] = result.get("elevation")
        time.sleep(1) 
    return elevation_map

elevation_map = get_elevations(unique_sites)
combined["Elevation (m)"] = combined["Site_Num"].map(elevation_map)

print(combined[["ACT_ID", "Site_Num", "Lat_Start", "Long_Start", "Elevation (m)", "Water_Body"]].drop_duplicates(subset="Site_Num").head())

                                            ACT_ID  Site_Num  Lat_Start  \
0    19830721-0-FFF-Piscassic River-Newfields-GILL         1    43.0343   
5        19830801-0-FFF-Lamprey River-Raymond-GILL         2    43.0385   
8          19830816-0-FFF-Cold River-Walpole-EFISH         3    43.1320   
69    19830824-0-FFF-Lamprey River-Deerfield-EFISH         4    43.1421   
293   19830824-0-FFF-Little River-Nottingham-EFISH         5    43.1498   

     Long_Start   Elevation       Water_Body  
0      -70.9681   28.565294  Piscassic River  
5      -71.1847   53.400249    Lamprey River  
8      -72.3896  115.555473       Cold River  
69     -71.2318  120.609673    Lamprey River  
293    -71.0671   55.864182     Little River  
